In [1]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

REPLICATE_API_TOKEN = getpass()
os.environ["REPLICATE_API_TOKEN"] = "r8_01YrzTehxtHci1pTUY0jbBit87BcqDI4Ubi3l"

ImportError: cannot import name 'get_simple_explainer_template' from 'prompting' (/share/u/caden/autointerp/single/prompting/__init__.py)

In [ ]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [ ]:
# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

Loaded dictionaries


In [ ]:
env = Environment(
    model=model, 
    sae_list=sae_list,
)

In [ ]:
env.load(
    layer=10,
    feature_id=7000,
)

 69%|██████▉   | 9/13 [00:22<00:12,  3.04s/it]

: 

In [7]:
long_explanation_list = env.explainer.explain()

env.state.history.append({"role":"system","message":long_explanation_list})

Processing batches: 100%|██████████| 2/2 [00:16<00:00,  8.09s/it]


In [16]:
from condenser import condense
from prompting import get_simple_condenser_template

explanation_list, output = condense(long_explanation_list, return_output=True)

env.state.history.append({"role":"user","message":get_simple_condenser_template(long_explanation_list)})
env.state.history.append({"role":"assistant","message":"".join(output)})

In [25]:
d_scores_list = env.d_scorer.score(explanation_list)
pruned_explanation_list = [explanation_list[i] for i in range(len(explanation_list)) if d_scores_list[i][0] > 0.6 and d_scores_list[i][1] < 0.3]

pruned_explanation_list

[(tensor([3155,  286,  883, 5341,  339,  772, 5201,  572,  530, 2366,  287,  262,
        7521,   11,  356,  389, 9389,  683]), tensor([ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  1.1186, 36.0348,
         0.0000,  0.0000])), tensor([ 460, 4886,  345,  460,  307,  355,  881,  355,  838, 1661, 2252,  618,
         345, 1183,  766,  683,  878,  339, 1183, 1683]), (tensor([2339, 9978,   13,  314,  716, 1654,  612,  318,  257, 1438,  329,  340,
          11,  290,  314,  716, 1654,  530]), tensor([ 0.0000,  0.0000,  0.0000,  0.0000, 21.1534,  0.8853,  0.0000,  0.0000,
         0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000, 33.9865,
         3.0725,  0.0000])), tensor([30098,  1878,   290, 19148,   430,   547,  1111, 38368,   771,   656,
        11327,    11,   511, 11316,  1719,   587, 19267,   351,   262,  3452]), tensor([  198,   198,   447,   250,  2215,   262,  1230,  2753,   661, 

TypeError: argument 'ids': 'list' object cannot be interpreted as an integer

In [ ]:
g_scores_list = env.gscorer.score(explanation_list)

In [ ]:
for i in range(len(explanation_list)):
  print(f"Explanation: {explanation_list[i]}")
  print(f"Detection rate: {d_scores_list[i][0]:.2f}")
  print(f"False positive rate: {d_scores_list[i][1]:.2f}")
  print(f"Generation score: {g_scores_list[i]:.2f}")
  print()